# ARR 2019 Areal Reduction Factors — a Python Port of Tony Ladson's R Method

**Companion notebook to:** _Areal Reduction Factors in Python — a Port of Tony Ladson's ARR 2019 Method_

**Source:** [ARR2019 – Areal Reduction Factors](https://tonyladson.wordpress.com/2020/04/05/arr2019-areal-reduction-factors/) and [Areal reduction factors – some edge cases](https://tonyladson.wordpress.com/2020/04/14/arr2019-areal-reduction-factors-some-edge-cases/) — Tony Ladson, April 2020. R source: [gist.github.com/TonyLadson/fc870cf7ebfe39ea3d1a812bcc53c8fb](https://gist.github.com/TonyLadson/fc870cf7ebfe39ea3d1a812bcc53c8fb) (`ARF2019.R`) and [gist.github.com/TonyLadson/b8baac6c450fe7f32f5020eb496e8b62](https://gist.github.com/TonyLadson/b8baac6c450fe7f32f5020eb496e8b62) (`ARF_edge_cases.R`).

This is a Python port of Ladson's complete ARR 2019 Book 2 Areal Reduction Factor implementation — both equations (short duration <12h, long duration 24-168h, 10 climatological regions), the interpolation logic for 12-24h *and* for small catchments (<10 km²) which isn't obvious from the ARR text alone, and validated directly against two exact values he prints in his own script.

**This was a genuinely stalled draft before this notebook.** An earlier pass at this post had the short-duration equation reconstructed (unverified) from a third-party spreadsheet vendor, and an explicit TODO for the long-duration equation, its 10 regional coefficients, and the small-area/duration-interpolation logic, because the ARR 2019 PDF wasn't accessible in that session. Finding Ladson's real, working implementation resolved all of it at once.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
print('numpy:', np.__version__)

numpy: 2.4.6


## 1. The 10 climatological regions

Long-duration ARFs use different coefficients (a-i, per ARR 2019 Book 2) for each of 10 regions, delineated by climatology (Podger et al. methodology, adopted into ARR 2019) -- not by state or administrative boundary.

In [2]:
REGIONS = {
    'East Coast North':     (0.327,  0.241, 0.448, 0.36,  0.00096,   0.48,   -0.21,  0.012,   -0.0013),
    'Semi-arid Inland QLD': (0.159,  0.283, 0.25,  0.308, 7.3e-07,   1.0,     0.039, 0.0,      0.0),
    'Tasmania':              (0.0605, 0.347, 0.2,  0.283, 0.00076,   0.347,   0.0877,0.012,   -0.00033),
    'SW WA':                 (0.183,  0.259, 0.271,0.33,  3.845e-06, 0.41,    0.55,  0.00817, -0.00045),
    'Central NSW':           (0.265,  0.241, 0.505,0.321, 0.00056,   0.414,  -0.021, 0.015,   -0.00033),
    'SE Coast':               (0.06,   0.361, 0.0, 0.317, 8.11e-05,  0.651,   0.0,   0.0,      0.0),
    'Southern Semi-arid':     (0.254,  0.247, 0.403,0.351, 0.0013,    0.302,   0.058, 0.0,      0.0),
    'Southern Temperate':     (0.158,  0.276, 0.372,0.315, 0.000141,  0.41,    0.15,  0.01,    -0.0027),
    'Northern Coastal':       (0.326,  0.223, 0.442,0.323, 0.0013,    0.58,   -0.374, 0.013,   -0.0015),
    'Inland Arid':            (0.297,  0.234, 0.449,0.344, 0.00142,   0.216,   0.129, 0.0,      0.0),
}
print(f'{len(REGIONS)} regions: {list(REGIONS)}')

10 regions: ['East Coast North', 'Semi-arid Inland QLD', 'Tasmania', 'SW WA', 'Central NSW', 'SE Coast', 'Southern Semi-arid', 'Southern Temperate', 'Northern Coastal', 'Inland Arid']


## 2. The two core equations

**Short duration (<=12h, one national equation):**

In [3]:
def arf_short(area, duration, aep):
    """ARR 2019 short-duration (<=720 min) Areal Reduction Factor.

    area : catchment area, km^2
    duration : storm duration, minutes
    aep : Annual Exceedance Probability as a decimal (e.g. 0.01 for 1%)
    """
    a, b, c, d = 0.287, 0.265, 0.439, 0.36
    e, f, g = 0.00226, 0.226, 0.125
    h, i, j = 0.0141, -0.021, 0.213
    val = (1
           - a * (area**b - c * np.log10(duration)) * duration**(-d)
           + e * area**f * duration**g * (0.3 + np.log10(aep))
           + h * area**j * 10**(i * (1/1440) * (duration - 180)**2) * (0.3 + np.log10(aep)))
    return min(1.0, val)

**Long duration (24-168h, 10 regional coefficient sets):**

In [4]:
def arf_long(area, duration, aep, region):
    """ARR 2019 long-duration (>=1440 min) Areal Reduction Factor."""
    a, b, c, d, e, f, g, h, i = REGIONS[region]
    val = (1
           - a * (area**b - c * np.log10(duration)) * duration**(-d)
           + e * area**f * duration**g * (0.3 + np.log10(aep))
           + h * 10**(i * area * duration / 1440) * (0.3 + np.log10(aep)))
    return min(1.0, val)

## 3. Validation against Ladson's own printed values

His `ARF_edge_cases.R` prints two specific values as commented-out output while investigating a real ARR 2019 quirk (see Section 5 below). Both are checked here directly, not eyeballed.

In [5]:
v_short = arf_short(26, 720, 0.0005)
v_long = arf_long(26, 1440, 0.0005, 'Tasmania')

print(f"arf_short(26, 720, 0.0005)             = {v_short:.7f}  (Ladson: 0.9377527)")
print(f"arf_long(26, 1440, 0.0005, 'Tasmania')  = {v_long:.7f}  (Ladson: 0.9322746)")

assert abs(v_short - 0.9377527) < 1e-6, 'short-duration mismatch'
assert abs(v_long - 0.9322746) < 1e-6, 'long-duration mismatch'
print()
print('Both match Ladson\'s published values to 7 decimal places.')

arf_short(26, 720, 0.0005)             = 0.9377527  (Ladson: 0.9377527)
arf_long(26, 1440, 0.0005, 'Tasmania')  = 0.9322746  (Ladson: 0.9322746)

Both match Ladson's published values to 7 decimal places.


## 4. The full dispatcher

The part that isn't obvious from the ARR 2019 text alone: what happens for catchments smaller than 10 km², and how the 12-24h interpolation actually works. Both come directly from Ladson's `ARF2019.R`.

For **duration >= 24h (long) or <= 12h (short) and area >= 10 km²**: use the relevant equation directly.

For **area between 1 and 10 km²** (either duration regime): compute the ARF at 10 km² first, then interpolate down using `ARF = 1 - 0.6614*(1 - ARF_at_10km2)*(area^0.4 - 1)` — not a detail that's obvious from the ARR text, and easy to miss entirely without a reference implementation.

For **duration between 12h and 24h**: linearly interpolate between the short-duration ARF at exactly 12h and the long-duration ARF at exactly 24h (both evaluated at the *target* area, if area >= 10 km²; at 10 km² first, then area-interpolated down, if area < 10 km²) — proportionally by how far the target duration sits between 720 and 1440 minutes.

**area <= 1 km²**: ARF = 1 (no reduction) in all cases.

In [6]:
def arf(area, duration, aep, region=None, neg_to_zero=True):
    """ARR 2019 Book 2 Areal Reduction Factor -- full dispatcher.

    Raises ValueError outside ARR 2019's stated validity range, rather
    than silently extrapolating: area in (0, 30000] km^2, aep in
    [0.005, 0.5], duration in [0, 10080] min (7 days). Short-duration
    equations are additionally invalid for area > 1000 km^2.
    """
    if not (0 < area <= 30000):
        raise ValueError('area must be between 0 and 30,000 km^2')
    if not (0.005 <= aep <= 0.5):
        raise ValueError('aep must be between 0.005 and 0.5 (0.5% to 50%)')
    if not (0 <= duration <= 7 * 24 * 60):
        raise ValueError('duration must be between 0 and 10,080 min (7 days)')
    if duration <= 720 and area > 1000:
        raise ValueError("Generalised equations aren't applicable for short durations when area > 1000 km^2")

    if area <= 1:
        return 1.0

    if duration >= 1440:
        if area >= 10:
            return arf_long(area, duration, aep, region)
        arf_long_10 = arf_long(10, duration, aep, region)
        return 1 - 0.6614 * (1 - arf_long_10) * (area**0.4 - 1)

    if duration <= 720:
        if area >= 10:
            if area > 1000:
                raise ValueError("Generalised equations aren't applicable for short duration events on areas > 1000 km^2")
            val = arf_short(area, duration, aep)
            return max(0.0, val) if neg_to_zero else val
        arf_short_10 = arf_short(10, duration, aep)
        return 1 - 0.6614 * (1 - arf_short_10) * (area**0.4 - 1)

    # 720 < duration < 1440: the 12h-24h interpolation zone
    if 1 < area < 10:
        arf_long_24 = arf_long(10, 1440, aep, region)
        arf_short_12 = arf_short(10, 720, aep)
        arf_interp_10 = arf_short_12 + (arf_long_24 - arf_short_12) * (duration - 720) / 720
        return 1 - 0.6614 * (1 - arf_interp_10) * (area**0.4 - 1)

    arf_long_24 = arf_long(area, 1440, aep, region)
    arf_short_12 = arf_short(area, 720, aep)
    return arf_short_12 + (arf_long_24 - arf_short_12) * (duration - 720) / 720

## 5. Checking continuity at the 12h and 24h boundaries

The dispatch logic above uses different code paths on either side of 720 and 1440 minutes. If the interpolation is implemented correctly, the ARF value shouldn't jump at either boundary.

In [7]:
print('Continuity at the 12h (720 min) boundary:')
for d in [719, 720, 721]:
    print(f'  duration={d:4d} min -> ARF={arf(500, d, 0.005, "Tasmania"):.4f}')

print()
print('Continuity at the 24h (1440 min) boundary:')
for d in [1439, 1440]:
    print(f'  duration={d:4d} min -> ARF={arf(30000, d, 0.005, "Tasmania"):.4f}')

print()
print('Small-area handling:')
print(f'  arf(0.5, 500, 0.01, "SE Coast")  = {arf(0.5, 500, 0.01, "SE Coast")}   (area<=1 -> exactly 1.0)')
print(f'  arf(5,   500, 0.01, "SE Coast")  = {arf(5, 500, 0.01, "SE Coast"):.4f}   (short, area<10 interpolation)')
print(f'  arf(5,  2000, 0.01, "SE Coast")  = {arf(5, 2000, 0.01, "SE Coast"):.4f}   (long, area<10 interpolation)')
print(f'  arf(5,   900, 0.01, "SE Coast")  = {arf(5, 900, 0.01, "SE Coast"):.4f}   (12-24h zone, area<10)')

Continuity at the 12h (720 min) boundary:
  duration= 719 min -> ARF=0.8523
  duration= 720 min -> ARF=0.8523
  duration= 721 min -> ARF=0.8524

Continuity at the 24h (1440 min) boundary:
  duration=1439 min -> ARF=0.6254
  duration=1440 min -> ARF=0.6256

Small-area handling:
  arf(0.5, 500, 0.01, "SE Coast")  = 1.0   (area<=1 -> exactly 1.0)
  arf(5,   500, 0.01, "SE Coast")  = 0.9788   (short, area<10 interpolation)
  arf(5,  2000, 0.01, "SE Coast")  = 0.9922   (long, area<10 interpolation)
  arf(5,   900, 0.01, "SE Coast")  = 0.9842   (12-24h zone, area<10)


## 6. A genuine ARR 2019 quirk, not a bug

Ladson's `ARF_edge_cases.R` documents something worth knowing before trusting an ARF curve blindly: for some catchments, the short-duration ARF at 12h is *larger* than the long-duration ARF at 24h — meaning the ARF-vs-duration curve briefly slopes *downhill* around the 12-24h transition, rather than monotonically decreasing with duration the way intuition suggests it always should.

In [8]:
short_12 = arf_short(26, 720, 0.0005)
long_24 = arf_long(26, 1440, 0.0005, 'Tasmania')
print(f'26 km^2 Tasmanian catchment, AEP=0.05%:')
print(f'  short-duration ARF at 12h = {short_12:.4f}')
print(f'  long-duration ARF at 24h  = {long_24:.4f}')
print(f'  short > long: {short_12 > long_24}  -- the curve is briefly non-monotonic here, and that\'s expected, not a bug.')

26 km^2 Tasmanian catchment, AEP=0.05%:
  short-duration ARF at 12h = 0.9378
  long-duration ARF at 24h  = 0.9323
  short > long: True  -- the curve is briefly non-monotonic here, and that's expected, not a bug.


## 7. Visualising the three zones

In [9]:
fig, ax = plt.subplots(figsize=(9, 6))
durations = np.unique(np.concatenate([np.logspace(0, np.log10(10080), 300), [719, 720, 721, 1439, 1440, 1441]]))
for area, color in [(100, '#4C72B0'), (1000, '#DD8452'), (26, '#55A868')]:
    aeps = []
    for d in durations:
        try:
            aeps.append(arf(area, d, 0.005, 'Tasmania'))
        except ValueError:
            aeps.append(np.nan)
    ax.plot(durations, aeps, color=color, lw=1.5, label=f'{area} km$^2$')

ax.axvline(720, color='grey', linestyle=':', alpha=0.5)
ax.axvline(1440, color='grey', linestyle=':', alpha=0.5)
ax.text(720, 0.05, '12h', ha='right', fontsize=8, color='grey')
ax.text(1440, 0.05, '24h', ha='left', fontsize=8, color='grey')
ax.set_xscale('log')
ax.set_xlabel('Duration (min, log scale)')
ax.set_ylabel('Areal Reduction Factor')
ax.set_title('ARR 2019 ARF vs. duration, Tasmania region, AEP=0.5%', fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('../../images/2026-09_arf-duration-curves.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size ... with Axes>

The 26 km² curve shows the non-monotonic dip right at the 12h-24h transition described in Section 6 — small enough to be easy to miss on a coarser plot, and exactly where Ladson's edge-case post says to expect it.

## 8. Usage: generating an ARF matrix for a design storm workflow

The kind of thing this is actually for — a table of ARFs across the durations and AEPs a URBS/RORB run needs, for one catchment.

In [10]:
AREA = 850  # km^2
REGION = 'East Coast North'
durations_h = [1, 2, 3, 6, 9, 12, 18, 24, 36, 48, 72]
aeps = [0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005]

print(f'{"Duration(h)":>12}', *[f'{a*100:>7.1f}%' for a in aeps])
for dh in durations_h:
    row = []
    for a in aeps:
        try:
            row.append(f'{arf(AREA, dh*60, a, REGION):7.3f}')
        except ValueError:
            row.append('    n/a')
    print(f'{dh:>12}', *row)

 Duration(h)    50.0%    20.0%    10.0%     5.0%     2.0%     1.0%     0.5%
           1   0.659   0.637   0.621   0.605   0.583   0.567   0.551
           2   0.741   0.712   0.691   0.669   0.641   0.619   0.598
           3   0.779   0.748   0.724   0.700   0.669   0.645   0.621
           6   0.833   0.816   0.804   0.791   0.774   0.762   0.749
           9   0.858   0.848   0.841   0.834   0.825   0.818   0.811
          12   0.873   0.864   0.857   0.850   0.840   0.833   0.826
          18   0.893   0.887   0.882   0.878   0.872   0.867   0.863
          24   0.913   0.910   0.908   0.906   0.904   0.902   0.900
          36   0.926   0.924   0.922   0.921   0.919   0.917   0.916
          48   0.934   0.933   0.931   0.930   0.928   0.926   0.925
          72   0.945   0.943   0.942   0.940   0.939   0.937   0.936


## References

- Ladson, A.R. (2020). [ARR2019 – Areal Reduction Factors](https://tonyladson.wordpress.com/2020/04/05/arr2019-areal-reduction-factors/); [Areal reduction factors – some edge cases](https://tonyladson.wordpress.com/2020/04/14/arr2019-areal-reduction-factors-some-edge-cases/). R source: [gist.github.com/TonyLadson/fc870cf7ebfe39ea3d1a812bcc53c8fb](https://gist.github.com/TonyLadson/fc870cf7ebfe39ea3d1a812bcc53c8fb), [gist.github.com/TonyLadson/b8baac6c450fe7f32f5020eb496e8b62](https://gist.github.com/TonyLadson/b8baac6c450fe7f32f5020eb496e8b62).
- Ball, J. et al. (2019). *Australian Rainfall and Runoff.* Book 2.